# 🧪 Plant Disease Detection — Model Prototyping and Experiments

This notebook demonstrates how to load the CNN models, run dummy forward passes to verify output shapes, and test prediction logic on a sample image.

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image

from src.model import get_model
from src.preprocessing import get_val_test_transforms
from src import config

## 🏗️ 1. Instantiating the Models

We load both the custom Baseline CNN and the ResNet-18 Transfer Learning model, checking their structures.

In [ ]:
num_classes = 5  # For prototype mode

baseline_model = get_model("baseline", num_classes=num_classes)
resnet_model = get_model("resnet18", num_classes=num_classes, pretrained=False)

print(f"Baseline CNN Model:\n{baseline_model}\n")
print(f"ResNet-18 Transfer Learning Model:\n{resnet_model}")

## ➡️ 2. Forward Pass with Dummy Data

We verify that passing a tensor of shape `(batch_size, 3, 224, 224)` through the models produces the expected logit output shape of `(batch_size, num_classes)`.

In [ ]:
dummy_input = torch.randn(4, 3, 224, 224)

baseline_out = baseline_model(dummy_input)
resnet_out = resnet_model(dummy_input)

print(f"Input shape: {dummy_input.shape}")
print(f"Baseline CNN Output shape: {baseline_out.shape} (Expected: [4, 5])")
print(f"ResNet-18 Output shape: {resnet_out.shape} (Expected: [4, 5])")

## 🎯 3. Image Preprocessing and Mock Inference

We run a full pre-processing and model execution sequence on a dummy image.

In [ ]:
# Create a dummy white image
dummy_image_path = "dummy_leaf.jpg"
img = Image.new("RGB", (300, 300), color=(255, 255, 255))
img.save(dummy_image_path)

# Apply validation transforms
transform = get_val_test_transforms()
img_tensor = transform(img).unsqueeze(0)  # Add batch dimension -> (1, 3, 224, 224)

print(f"Preprocessed image shape: {img_tensor.shape}")

# Forward pass
resnet_model.eval()
with torch.no_grad():
    logits = resnet_model(img_tensor)
    probs = torch.softmax(logits, dim=1)
    
pred_class = torch.argmax(probs, dim=1).item()
confidence = probs[0, pred_class].item()

print(f"Predicted Class Index: {pred_class}")
print(f"Confidence: {confidence:.2%}")

# Clean up dummy image
if os.path.exists(dummy_image_path):
    os.remove(dummy_image_path)